# Baseline CNN Reproduction (Scoliosis1K)

**Paper Reference:** Table 3, Row 1 (Accuracy: 45.5%, Sensitivity: 99.0%, Specificity: 27.0%).

**Paper Definition of Baseline CNN (§4.3):**
> *"A basic model trained with cross-entropy loss, excluding horizontal pooling and BNNeck."*

**Implementation Choices to Match Paper:**
- **Backbone**: OpenGait `ResNet9` (channels `[64, 128, 256, 512]`, strides `[1, 2, 2, 1]`).
- **Temporal Aggregation**: Temporal Max Pooling (TP) over frames.
- **Head**: Global Average Pooling -> Linear(512, 256) -> Linear(256, 3). *No BNNeck, no scale=16.*
- **Loss**: Cross-Entropy only. Label smoothing 0.1 (OpenGait default).
- **Optimizer**: SGD, LR 0.1, Momentum 0.9, Weight Decay 0.0005. Milestones at 10k, 14k, 18k.
- **Data Pipeline**: `TripletSampler` (8 identities x 8 sequences) over the 1:1:8 split from `Scoliosis1K_118.json`.

**Expected Results Note**: The test set is heavily skewed toward Positive (419 out of 749). Blind guessing "Positive" yields ~56% accuracy. The paper's Baseline CNN achieving 45.5% is expected for a CE-only model without triplet identity supervision. It is completely normal if your reproduced accuracy lands in the 40-55% range.

In [ ]:
import copy, json, pickle, random, time
from functools import lru_cache
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ==========================================
# 1. SETTINGS
# ==========================================

# UPDATE THESE PATHS TO MATCH YOUR LOCAL SETUP
DATASET_ROOT = Path(r"C:\Users\arina\OneDrive\Documents\Reproduce_Paper\OpenGait\datasets\Scoliosis1K-sil-pkl\Scoliosis1K-sil-pkl")
PARTITION_JSON = Path(r"C:\Users\arina\OneDrive\Documents\Reproduce_Paper\OpenGait\datasets\Scoliosis1K\Scoliosis1K_118.json")

LABELS = ["negative", "neutral", "positive"] # Class IDs 0, 1, 2
LABEL_TO_ID = {name: i for i, name in enumerate(LABELS)}
NUM_CLASSES = 3
N_FRAMES = 30

# Paper Hyperparameters
TOTAL_ITERS = 20000
MILESTONES = [10000, 14000, 18000]
TRAIN_LR = 0.1
IDS_PER_BATCH = 8
SEQS_PER_ID = 8
LABEL_SMOOTHING = 0.1

# For quick smoke-testing the pipeline, set True. MUST be False for paper reproduction.
DEBUG_SINGLE_BATCH = False

SEED = 0
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")
print(f"Dataset Root: {DATASET_ROOT.resolve()}")

In [ ]:
# ==========================================
# 2. DATA LOADING & PREPROCESSING
# ==========================================
def load_pkl(path):
    with open(path, "rb") as f:
        seq = np.asarray(pickle.load(f))
    if seq.ndim == 4 and seq.shape[1] == 1:
        seq = seq[:, 0]
    elif seq.ndim == 4 and seq.shape[-1] == 1:
        seq = seq[..., 0]
    return seq

def cut_and_normalize(seq):
    cut = seq.shape[-1] // 64 * 10
    if cut > 0:
        seq = seq[..., cut:-cut]
    return seq.astype(np.float32) / 255.0

def random_clip(seq, n_frames=N_FRAMES):
    idx = np.random.choice(len(seq), n_frames, replace=len(seq) < n_frames)
    return seq[idx]

def train_clip(seq, n_frames=N_FRAMES):
    return cut_and_normalize(random_clip(seq, n_frames))

@lru_cache(maxsize=None)
def load_pkl_cached(path):
    return load_pkl(path)

def build_index(root, subject_ids):
    items = []
    for sid in subject_ids:
        folder = root / str(sid)
        if not folder.is_dir(): continue
        for pkl in sorted(folder.rglob("*.pkl")):
            names = [p.lower() for p in pkl.relative_to(root).parts[:-1]]
            class_id = next((LABEL_TO_ID[n] for n in names if n in LABEL_TO_ID), None)
            if class_id is not None:
                items.append((str(sid), class_id, pkl))
    return items

def make_triplet_batch_fn(items, ids_per_batch, seqs_per_id):
    by_subject = {}
    for sid, class_id, path in items:
        by_subject.setdefault(sid, []).append((class_id, path))
    subjects = list(by_subject)
    index = {sid: i for i, sid in enumerate(subjects)}

    def get_batch():
        clips, ids, classes = [], [], []
        for sid in random.sample(subjects, min(ids_per_batch, len(subjects))):
            pool = by_subject[sid]
            if len(pool) >= seqs_per_id:
                picks = random.sample(pool, seqs_per_id)
            else:
                picks = random.choices(pool, k=seqs_per_id)
            for class_id, path in picks:
                clips.append(train_clip(load_pkl_cached(path)))
                ids.append(index[sid])
                classes.append(class_id)
        return np.stack(clips), np.array(ids), np.array(classes)
    return get_batch

In [ ]:
# ==========================================
# 3. BASELINE CNN ARCHITECTURE
# ==========================================
class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = None
        if stride != 1 or in_ch != out_ch:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch))

    def forward(self, x):
        identity = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)

class ResNet9(nn.Module):
    def __init__(self, in_channels=1, channels=(64, 128, 256, 512), strides=(1, 2, 2, 1)):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, channels[0], 3, 1, 1, bias=False),
            nn.BatchNorm2d(channels[0]), nn.ReLU(inplace=True))
        blocks, prev = [], channels[0]
        for ch, stride in zip(channels, strides):
            blocks.append(BasicBlock(prev, ch, stride))
            prev = ch
        self.blocks = nn.Sequential(*blocks)

    def forward(self, x):
        return self.blocks(self.stem(x))

class FrameWise(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, x):
        n, c, s, h, w = x.shape
        x = x.permute(0, 2, 1, 3, 4).reshape(n * s, c, h, w)
        out = self.net(x)
        _, c2, h2, w2 = out.shape
        return out.reshape(n, s, c2, h2, w2).permute(0, 2, 1, 3, 4)

class BaselineCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.backbone = FrameWise(ResNet9())
        self.embedding = nn.Linear(512, 256)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, sils):
        x = sils.unsqueeze(1)       # [n, 1, s, h, w]
        x = self.backbone(x)        # [n, 512, s, 16, 11]
        x = x.max(dim=2)[0]         # Temporal Max Pooling: [n, 512, 16, 11]
        x = x.mean(dim=(2, 3))      # Global Spatial Average: [n, 512]
        embed = self.embedding(x)   # [n, 256]
        logits = self.classifier(embed) # [n, 3]
        return embed.unsqueeze(-1), logits.unsqueeze(-1)

def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.xavier_uniform_(m.weight)
    elif isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)) and m.affine:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

def make_model():
    model = BaselineCNN()
    model.apply(init_weights)
    return model.to(DEVICE)

In [ ]:
# ==========================================
# 4. TRAINING LOOP (CE ONLY)
# ==========================================
def compute_loss(logits, classes, label_smoothing=0.1):
    logits = logits.mean(-1)
    return F.cross_entropy(logits, classes, label_smoothing=label_smoothing)

def save_checkpoint(path, model, optimizer, scheduler, scaler, it, history):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                "iter": it, "history": history}, path)

def load_checkpoint(path, model, optimizer, scheduler, scaler):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    print(f"Resumed from {path} at iteration {ckpt['iter']}")
    # Safety check for partner's feedback #3
    assert scheduler.last_epoch == ckpt["iter"], f"Scheduler mismatch! Expected {ckpt['iter']}, got {scheduler.last_epoch}"
    return ckpt["history"], ckpt["iter"] + 1

def train(model, get_batch, iters, lr=0.1, milestones=(), log_every=100, amp=False, 
          label_smoothing=0.1, ckpt_path=None, save_every=5000, resume=False):
    amp = amp and DEVICE.type == "cuda"
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=list(milestones), gamma=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    history = {"ce": [], "acc": []}
    start_it = 1

    if resume and ckpt_path and Path(ckpt_path).is_file():
        history, start_it = load_checkpoint(ckpt_path, model, optimizer, scheduler, scaler)

    t0 = time.time()
    for it in range(start_it, iters + 1):
        model.train()
        sils, ids, classes = get_batch()
        sils = torch.as_tensor(sils, dtype=torch.float32, device=DEVICE)
        classes = torch.as_tensor(classes, dtype=torch.long, device=DEVICE)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=amp):
            embed, logits = model(sils)
        logits = logits.float()
        loss = compute_loss(logits, classes, label_smoothing)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        acc = (logits.mean(-1).argmax(1) == classes).float().mean().item()
        history["ce"].append(loss.item())
        history["acc"].append(acc)

        if it == start_it or it % log_every == 0 or it in milestones:
            elapsed = time.time() - t0
            eta = elapsed / (it - start_it + 1) * (iters - it)
            lr_now = optimizer.param_groups[0]['lr']
            print(f"iter {it:5d} | LR {lr_now:.2e} | CE {loss.item():.4f} | acc {acc:.2f} | ETA {eta/60:.1f}m")
            
        if ckpt_path and save_every and (it % save_every == 0 or it == iters):
            save_checkpoint(ckpt_path, model, optimizer, scheduler, scaler, it, history)
    return history

In [ ]:
# ==========================================
# 5. EVALUATION
# ==========================================
@torch.no_grad()
def predict(model, seq_uint8, max_frames=720):
    model.eval()
    seq_uint8 = seq_uint8[:max_frames]
    x = torch.from_numpy(cut_and_normalize(seq_uint8)).unsqueeze(0).to(DEVICE)
    _, logits = model(x)
    scores = logits.mean(-1)
    probs = torch.softmax(scores, dim=-1)[0].cpu().numpy()
    return int(probs.argmax()), probs

def confusion(model, items, max_frames=720):
    conf = np.zeros((3, 3), dtype=int)
    for _, class_id, path in items:
        pred, _ = predict(model, load_pkl_cached(path), max_frames)
        conf[class_id, pred] += 1
    return conf

def paper_metrics(conf):
    accuracy = conf.trace() / max(conf.sum(), 1)
    sensitivity = conf[0, 0] / max(conf[0].sum(), 1) # Negative Recall
    specificity = conf[1:, 1:].sum() / max(conf[1:].sum(), 1) # Pos+Neu Recall
    return accuracy, sensitivity, specificity

def evaluate(model, items):
    conf = confusion(model, items)
    acc, sen, spe = paper_metrics(conf)
    print("\nConfusion Matrix (Rows: True [Neg, Neu, Pos], Cols: Pred):")
    print(conf)
    print(f"\nMetrics -> Accuracy: {acc:.1%} | Sensitivity: {sen:.1%} | Specificity: {spe:.1%}")
    print("Paper Table 3 -> Accuracy: 45.5% | Sensitivity: 99.0% | Specificity: 27.0%")

In [ ]:
# ==========================================
# 6. EXECUTE TRAINING & EVAL
# ==========================================
CKPT_PATH = "baseline_cnn_ce.pt"
RUN_FULL = True

if not RUN_FULL:
    print("RUN_FULL is False. Skipping full training.")
else:
    with open(PARTITION_JSON, "r", encoding="utf-8") as f:
        partition = json.load(f)
        
    train_items = build_index(DATASET_ROOT, partition["TRAIN_SET"])
    test_items = build_index(DATASET_ROOT, partition["TEST_SET"])
    
    print(f"Train sequences: {len(train_items)} | Test sequences: {len(test_items)}")
    if DEBUG_SINGLE_BATCH:
        print("WARNING: DEBUG_SINGLE_BATCH is True. This is for pipeline testing only, NOT the paper protocol.")
        train_items = [next(i for i in train_items if i[1] == c) for c in range(3)]
        print(f"Reduced train set to 3 sequences (one per class).")
    
    set_seed()
    model = make_model()
    get_batch = make_triplet_batch_fn(train_items, IDS_PER_BATCH, SEQS_PER_ID)
    
    print(f"\nStarting training: {TOTAL_ITERS} iterations, {IDS_PER_BATCH}x{SEQS_PER_ID} batches, LR {TRAIN_LR}")
    history = train(
        model, get_batch, iters=TOTAL_ITERS, lr=TRAIN_LR, 
        milestones=MILESTONES, log_every=100, amp=True, 
        label_smoothing=LABEL_SMOOTHING, ckpt_path=CKPT_PATH, save_every=5000, resume=False
    )
    
    print("\nTraining Complete. Evaluating on full test set...")
    evaluate(model, test_items)